# Natural Language Processing and Topic Modelling

The aim of this workshop is to introduce you to the application of natural language processing (NLP) algorithms to study free text data. NLP is a branch of artificial intelligence that focuses on trying to understand written and spoken text. NLP is a very active area of research with new methods being developed all time. In this workshop, we will focus specifically on topic modelling, which is an unsupervised machine learning method that analyses a corpus of documents that people have written on one or other theme in order to identify the most common topics within that theme.

In particular, we will use a type of topic modelling called **Latent Dirchlet Allocation (LDA)** to examine the abstracts from 2067 papers authored by researchers within the MDR (Metabolism Digestion and Reproduction) department to see whether we can identify common themes of research, and to find out which theme best describes the research of some authors/division/sections of your interest.

Concretely, in this workshop, we will learn the following:
1. Data cleaning and pre-processing of free text data (e.g., abstracts from papers)
3. Topic modelling using Latent Dirichlet Allocation
2. Visualisation of free text data using Wordclouds



The first thing we need to do is to import the packages we will need during the workshop, and to change the display settings in order to be able to visualise more rows and columns when printing dataframes. Today, we will work a lot with two new python modules called gensim and nltk. Gensim is one of the most commonly used modules for topic modelling in Python, and nltk is an NLP Python toolkit. 

In [ ]:
import json
import pandas as pd
from tqdm import tqdm
import ast
import os

import pandas as pd
import warnings 
import gensim
import nltk
import spacy
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', 500)

sb.set_theme("talk")
sb.set_style("whitegrid")


We will start by importing the data, which are stored in a json file. The data are imported in the form of a dictionary with three keys: `source`, `date` and `documents`. The document key corresponds to a list of dictionaries where each dictionary contains information about one paper. Specifically, each paper is associated to an `id`, `doi`, `pmid`, `title`, `abstract`, `author`, `division`, `section` and `keywords`. To make the following steps of the analysis easier, we will convert these dictionaries into a dataframe, where each row corresponds to a paper.

In [ ]:
f = open('PubMed_MDR_2026_v2.json', encoding="utf-8")
json1_str = f.read()
data = json.loads(json1_str)

print("Keys in the data:", list(data.keys()))
print("Information about each paper:", list(data['documents'][0].keys()))


In [ ]:
data_df = pd.DataFrame.from_dict(data["documents"])
data_df.head()

    Can you find out how many divisions and sections are included in the first paper?

In [ ]:
# CODE HERE TO INDEX THE FIRST ELEMENT AND EXTRACT THE DIVISION AND SECTION INFORMATION



Before moving forwards with the analysis, you can pick any of these questions to answer. Follow the instructions based on the question you selected!
1. **What are the main research lines (topics, keywords) pursued by MDR as a whole?**: you can move to the **Data Cleaning and pre-processing** section without the need of any filtering (NOTE: this will take considerable time during this tutorial, suggestion is to focus on a smaller subset!)
2. **What are the main research lines (topics, keywords) pursued by each division?**: filter the data using the `filter_data` function, set `level_of_filtering`equal to **division** and `name_of_interest`to the division you want to work on
3. **What are the main research lines (topics, keywords) pursued by each section?**: filter the data using the `filter_data` function, set `level_of_filtering`equal to **section** and `name_of_interest`to the section you want to work on
4. **What are the main research lines (topics, keywords) pursued by my 3-5 authors?**: filter the data using the `filter_data` function, set `level_of_filtering`equal to **author** and `name_of_interest`to the author you want to work on or the list of authors you want to work on

In [ ]:
def filter_data(data_df, level_of_filtering, name_of_interest):
    rows_to_keep = []
    for i, list_names in enumerate(data_df[level_of_filtering]):
        if isinstance(name_of_interest, list):
            for name in name_of_interest:
                if name in list_names:
                    rows_to_keep.append(i)
        else:
            if name_of_interest in list_names:
                rows_to_keep.append(i)
    rows_to_keep = list(set(rows_to_keep))     
    data_df = data_df.iloc[rows_to_keep]
    return data_df

In [ ]:
# CODE HERE: Filter your dataframe here
# EXAMPLE 1: data_df = filter_data(data_df, "division", 'Systems Medicine')
# EXAMPLE 2: data_df = filter_data(data_df, "author", ['Temelkuran B', 'Lees C'])


In [ ]:
data_df = data_df.reset_index(drop = True)

# Data cleaning and pre-processing

By simply looking at the first rows of the dataframe, you will immediately see that the data are not ready to be used for analysis. For example, the abstracts still have special characters or some words are written with capital letters and others not... All these aspects are just few examples of the noise that free text documents have, and that needs to be removed before starting any analysis. The data cleaning step in free text analysis is **fundamental**! 

Data cleaning is necessary to remove errors in the data, to reduce the noise to the minimum and to include in the analysis only what is essential. The most important cleaning steps of free-text data are the following:

1. **Turning all letters to lower case**: this is important otherwise words with capital letters will be mistakenly recognised as different compared to the same words without capital letters (e.g. This and this).
2. **Removal of punctuation, special characters and digits**: punctuation creates noise in the data. It cannot be used to make sense of the meaning of a topic because it does not represent words and computers do not know how to interpret it.
3. **Tokenisation**: method that consists of segmenting a piece of text (in this case the abstracts) into its discrete units (i.e., words).
4. **Stop words removal:** method that consists of removing words that are really common in english, but don't provide much information about the content of a document when considered alone, e.g., "to", "in" or "when". These words just create noise in the data for a method like LDA, which is a type of topic modelling (though they can be important for other methods that consider the interrelationship/ordering of multiple words).
5. **Lemmatisation:** process of grouping together the inflected forms of a word so they can be analysed as a single item. E.g., run, running, runs, ran all have the same root and can be replaced with a single token (e.g, run).

Let's complete these cleaning steps, in order to understand better what they do and why they are important. First, we will turn everything to lower case.

In [ ]:
data_df['abstract'] = data_df['abstract'].str.lower() 
# Let's check a random abstract to see if something is still in capital letters
data_df['abstract'][1]

Then, we remove all punctuation (e.g. , or .), special characters (e.g. ?,/ and &) and digits.

In [ ]:
data_df['abstract'] = data_df['abstract'].str.replace('[()-\[,\.%!?/&\]]', '', regex = True)
data_df['abstract'] = data_df['abstract'].str.replace('\d+', '')
data_df['abstract'][1]

As you can see, now the punctuation and the digits are fully removed. 

The next step of the data cleaning and preparation is **tokenisation**. Tokenisation is necessary to make the sentences analysable and understandable for the computer, and consists of splitting each abstract into lists of individual words. There is a function in the `gensim` package that can do this directly, by taking as input each separate abstract, called `simple_preprocess``
. By providing as argument `deacc=True`, the function removes punctuations if it finds any. In addition, the function removes all words that have less that 3 letters or more than 20. Words with less than 3 letters are usually not really meaningful on their own, and those with more than 20 are suspiciously long in most normal contexts! 

In [ ]:
datalist = data_df['abstract'].to_list()
data_words = []
for sentence in datalist:
    listwords = gensim.utils.simple_preprocess(str(sentence), deacc=True, min_len=3, max_len=20)
    data_words.append(listwords)


In [ ]:
print(data_words[0])

Check different elements in the data_words list. Do you understand how tokenisation works?

Now that we have the abstracts in terms of lists of words, we can do some cleaning on the words themselves. The first thing we are going to do is to remove the stop words, or commonly used words in the English language. Luckily, the nltk module has alredy a list of these words that we can simply download.

In [ ]:
nltk.download('stopwords')
stop = nltk.corpus.stopwords.words('english') 

print(len(stop))

Of course, if you think that other words are too common and should be removed, but they are not in this list, you can easily add them. Adding an extra filter is always a good idea. In this way, the noise in the data is reduced even more. Do you have any other words in mind? If you do, add it to the following list.

In [ ]:
custom_stop = ['to', 'with', 'and', 'aim'] # could add words such as study and result here too if you want, but better to analyse later on if we need this
finalstop = stop + custom_stop
print(finalstop)

As you can see the words `to`, `and`and `with` were added at the end of the list. Now that we have our final list of stop words, we can remove those words from the list of words of each abstract.

In [ ]:
data_words_final = []

for abstract in data_words:
    data_words_cleaned = []
    for word in abstract:
        if word not in finalstop: # and word in words_english:
            data_words_cleaned.append(word)
    data_words_final.append(data_words_cleaned)

print("Before:" , data_words[2] ) 
print("\n")
print("After:" , data_words_final[2]  ) 

As you can see, words like `the`and `for` were removed. Now, we can complete the final step of the pre-processing, which is the **lemmatisation** step. Lemmatisation groups together different inflected forms of the same word, so that they can be analysed as being a single item with the same meaning. For example, it converts the different conjugations of verbs into the inifinite forms (e.g. "swim", "swam" and "swum" would be all converted to "swim"), or turns the plurals of nouns into the singular form.

In Python, there are different ways to complete the lemmatisation step. One way is to use the `nltk` package, and specifically, the `WordNetLemmatizer()` method and the `lemmatize` function. Another approach consists of using the package `spacy`. Let's try both approaches and see what's the difference.

### Lemmatisation with nltk

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')
lemma = nltk.stem.wordnet.WordNetLemmatizer()

In [ ]:
data_words_lemmatised = []
for abstract in data_words_final:
    data_words_cleaned = []
    for word in abstract:
        lematised = lemma.lemmatize(word)
        data_words_cleaned.append(lematised)
    data_words_lemmatised.append(data_words_cleaned)

print("Before:" , data_words_final[10]  ) 
print("\n")
print("After:" , data_words_lemmatised[10]  ) 

### Lemmatisation with spacy

`spacy` takes as input the original string of abstracts instead of the tokens. Therefore, we will first complete the lemmatisation, then remove the stop words and the words shorter than 3. 

In [ ]:
nlp = spacy.load("en_core_web_sm")
data_words_lemmatised = []

for abstract in tqdm(datalist):
    lemma_abstracts = []
    
    conference_help_doc = nlp(abstract)
    for token in conference_help_doc:
        if str(token) not in finalstop:
            if token.lemma_.strip() != "" and len(str(token.lemma_)) > 3:
                lemma_abstracts.append(token.lemma_)
    data_words_lemmatised.append(lemma_abstracts)

In [ ]:
print("Before:" , data_words_final[10]  ) 
print("\n")
print("After:" , data_words_lemmatised[10]  ) 

    Do you see any difference between the two approaches? Try to compare the two outputs and see whether the lemmatisation was completed differently for some words!

In [ ]:
# CODE HERE TO LOOK AT THE DIFFERENCES BETWEEN THE TWO LEMMATISATION METHODS


In [ ]:
clean_data = data_words_lemmatised.copy()

# Topic modelling using LDA
In today's workshop, we will do topic modelling using **Latent Dirichlet Allocation (LDA)**. LDA is amongst the most established and popular topic modelling methods (though there are new ones beng developed all the time), which aims to find the topics within a body of text based on the words it contains. Let's look at the name of the method and try to understand what each word means:

1. **Latent:** indicates that the model discovers the ‘yet-to-be-found’ or hidden topics that are common across the documents that people have written.
2. **Dirichlet:** indicates the two assumptions of LDA - that both the distribution of topics within a document and the distribution of words within each topic are Dirichlet distributions (which is a type of probability distribution).
3. **Allocation:** indicates the distribution of topics in the document.
LDA assumes that the words within a document can be used to determine the topics. LDA assigns each word in a document to a different topic, then maps the entire document to a list of topics. Put another way, LDA computes a many-to-many relationship between topics and words, and thus a many-to-many relationship between documents and topics. The many to many mapping of latent and written documents makes intuitive sense, as a document that someone has written can be saying the same thing as the one someone else wrote in somewhat different ways, but an individual document may also cover more than one topic.

One of the requirements of LDA is to specify the number of topics that should be identified within the set of abstracts. The number of topics is a user-defined parameter, better called a **hyperparamerter**. In machine learning, a hyperparameter is a parameter that is external to the model, that cannot be inferred from the data during a single run of the fitting process, and therefore needs to be fine-tuned each time depending on the model you are developing and the dataset you are using across multiple runs or based on prior knowledge in order to find the optimal or most approriate value respectively.

In the case of LDA, one of the best approaches to identify the optimal value for the number of topics is to use the **Coherence Score**. The coherence score specifies whether a certain topic split gives rise to coherent topics, where "coherent" is defined as combinations of documents that humans would tend to agree should be grouped together. The higher is the score, the more coherent are the topics. In order to use this score to identify the optimal number of topics, it is necessary to run the LDA analysis with different number of topics (this being the hyperparameter) and calculate the score for each of them. The number of topics that leads to the highest coherence score will correspond to the optimal number, which is then used as the optimal hyperparameter value in the model that we analyse/interpret.

**However, the coherence score metric is not perfect! First, it is partly stochastic, meaning that every time you run it, you could potentially get different results. Then, based on the hyperparameters you choose, you could get really different results. Finally, it can be really slow to run!! If you have any issues running the coherence score fine-tuning, let us know!**

Let's start by getting the data in the right format in order to be able to run LDA. The LDA function takes as input two main arguments:

1. A dictionary that has as keys an id number and as values a word (each word is assigned to a different id number)
2. The corpus which is essentially the list of abstracts in a bag-of-words format, that corresponds to (word_id, word_count), where the word id corresponds to the id assigned to the word in the dictionary.

To better understand what these are, let's create them using the `gensin` package.

In [ ]:
id2word = gensim.corpora.Dictionary(clean_data)

print("Total number of words:", len(id2word.keys()))
print(id2word[2])
print(id2word[3])
print(id2word[11])

As you can see id2word is a dictionary, where each key is a number, and the value of that key is a different word. For example, id 2 is assigned to the word `aggregation` (when it was run). In total, there are over 27K words across all the abstracts studied (you should have less when you are looking at a subset, and your second word is likely different).

In [ ]:
corpus = []
for abstract in clean_data:
    corpus.append(id2word.doc2bow(abstract))

print("Abstract", clean_data[0])
print("\n")
print("Corpus", corpus[0])

As you can see, each word of each abstract is converted into `(word_id, word_count)`, where the `word_id` is derived from the `id2word` dictionary

Now that we have the data in the right format, we can define which numbers of topics we want to test to find what the optimal number is. Today, we will try a maximum of 10 topics. Some data might require many more than 10, but the more topics numbers you test, the more time and computational resources you will need.

In [ ]:
min_topics = 1
max_topics = 11
step_size = 1
topics_range = range(min_topics, max_topics, step_size)
topics_range

**Let's run LDA with all the potential numbers of topics!**

To be able to do it, we need to complete the following steps:

1. Create a results dictionary where we will store the topic number and coherence score at each iteration
2. Loop over all the potential number of topics
3. Run the LDA model and change the number of topics at each iteration
4. Calculate the coherance score for each topic number
5. Save the coherence score in the results dictionary

**NOTE: Depending on the amount of memory that your computer has, this step can be either very slow or somewhat slow (yes it takes time no matter what - on a 512GB RAM machine (using 20 of its 48 cores) running up to 15 topics took half an hour for the ENTIRE dataset). To have an overview of how long you still need to wait before the computation is completed, Python has a really nice function and module called `tqdm` that creates progress bars when running for loops. If it is really slow, give a shout (but before you do, did you select a subset of the data?)**

In [ ]:
results = {'Topics_Number': [], "cv_Coherence_avg": []}
# finds number of CPU cores on current machine
nCores = np.min([os.cpu_count() - 1, 20])

for n_topics in tqdm(topics_range):
    lda_model = gensim.models.ldamulticore.LdaMulticore(corpus=corpus,
                                                        id2word=id2word,
                                                        num_topics=n_topics, 
                                                        random_state=100,
                                                        chunksize=100,
                                                        passes=50,
                                                        workers=nCores,
                                                        iterations=150,
                                                        minimum_probability=0)
                                                                 
    cv_coherence_total = gensim.models.CoherenceModel(model=lda_model, 
                                            texts=clean_data, 
                                            dictionary=id2word).get_coherence()
            
    results['Topics_Number'].append(n_topics)
    results['cv_Coherence_avg'].append(cv_coherence_total)

In [ ]:
results_df = pd.DataFrame(results)
results_df

Great! Now that we have the coherence score for each number of topics, we can create a plot to visualise the results and better see where the peak is.

In [ ]:
sb.lineplot(x='Topics_Number', y='cv_Coherence_avg', data=pd.DataFrame(results))
plt.xlabel("Number of topics")
plt.ylabel("Coherence Score")

Where is the peak in your plot? The peak corresponds to the optimal number of topics. 
However, to properly understand whether the number of topics is optimal, it is necessary to manually look at the final results, and see whether the identified topics make sense using a more qualitative approach. Using current methods, a combination of a quantitative (e.g., the coherence score) and qualitative approach is usually the best way to go, though the field is advancing quickly. Now that we know what the optimal topic number is, let's run LDA using the optimal number of topics.

In [ ]:
optimal_topic_number = results_df[results_df.cv_Coherence_avg == np.max(results_df.cv_Coherence_avg)].Topics_Number.item()

lda_model = gensim.models.ldamulticore.LdaMulticore(corpus=corpus,
                                                    id2word=id2word,
                                                    num_topics=optimal_topic_number, 
                                                    random_state=100,
                                                    chunksize=100,
                                                    passes=50,
                                                    workers=20,
                                                    iterations=150,
                                                    minimum_probability=0)
# Here we used a parallelised (using multiple CPU cores) implementation that speeds up topic model training

Let's look at the identified topics and the top 10 words that represent them using the method `show_topics`

In [ ]:
lda_model.show_topics(formatted=False, num_words= 10)

As you can see, the output of the function is the topic number, and a list of (word, probability) for each topic, where the probability corresponds to the probability of each word for that specific topic. Let's print only the words and ignore the probabilities for a moment.

In [ ]:
for idx, topic in lda_model.show_topics(formatted=False, num_words= 10):
    wordskeep = [w[0] for w in topic]
    print(f'Topic: {idx} \nWords: {wordskeep}')

### How would you describe each topic in one sentence? If you struggle to do it, try to increase the number of words that you print for each topic. Is the topic easier to understand now? Do the topics make sense?
Also, if you see words you think should be removed, you could add them to the stop word list and rerun from there.

# Wordclouds
Printing the words is of course useful, but there are better ways to visualise the most relevant words for each topic. One way is through **wordclouds**, which are a visual representation of the probability of words for each topic. The bigger is the word in the wordcloud, the higher is the probability for that word in the topic. Wordclouds can also be used to visualise the frequency of the words instead of the probaility derived from LDA. In that case, you would need to interpret the meaning of the wordcloud slightly differently.

## Wordclouds of LDA probability 

Now that we have the package, we can start preparing the input to the wordclouds function. We will first check how to create the wordcloud for one topic. To create a wordcloud for all topics, you can write a for loop and generate a figure with a subplot per topic. The input format of the wordclouds function is a dictionary, where each key is a word and the value is the probability. To obtain this dictionary, we can use a method called `show_topic`, which returns the words and the probability value for a specific topic. The `topn` parameter specifies how many top words should be selected for each topic. This parameter can be modified to control for how many words will appear in the wordcloud.

In [ ]:
topic_number = 5
words_topic = {}
for word, probability in lda_model.show_topic(topic_number, topn = 100): #0 means that we are extracting the words, probability for topic 0
    words_topic[word] = probability
print(words_topic)

In [ ]:
wc = WordCloud(background_color="white", max_words=50)
wc.generate_from_frequencies(words_topic)

_ = plt.figure(figsize = (10, 10))
plt.imshow(wc)
plt.axis("off")


Now, let's try to visualise the wordclouds for the other topics. To achieve this, you can either change the `topic_number` variable above, or write a for loop and generate one figure with different subplots (one per topic).

In [ ]:
# ADD YOUR CODE HERE TO PLOT ALL TOPICS


### Can you provide a better description of each topic using the wordclouds?
Do they relate to anything? Share your topics and descriptions in the Teams chat.

## Wordclouds of keywords

Wordclouds can be used to visualise different things. For example, instead of visualising the words with the highest probability for each topic, you could visualise how often each word appears across all abstracts, in a subset of abstracts or in the abstracts that are mostly described by a specific topic. This visualisation will allow you to identify the **keywords** of different topics. To be able to visualise the most common words across the abstracts that are best described by a specific topic, we first need to understand the probability of each topic for each abstract, adn then identify the predominant topic in each abstract.

To achieve this, you can use the `get_document_topics` function. 

In [ ]:
all_topics = lda_model.get_document_topics(corpus, minimum_probability=0.0)

print("Abstract", clean_data[10])
print("\n")
print(all_topics[10])

### Based on how you described each topic so far, does it make sense that this abstract was assigned to this topic?
    
Now that we have this information for each abstract, we want to extract it in an easier to view format, which is a dataframe. The aim is to create a dataframe where each row corresponds to an abstract, and each column represents the probability of each topic for that abstract. To find out the predominant topic for each abstract, we just have to identify the topic with the highest probability.

In [ ]:
all_topics_df = pd.DataFrame(gensim.matutils.corpus2csc(all_topics).T.toarray())
all_topics_df['dominant_topic_contribution'] = all_topics_df.max(axis = 1) 
all_topics_df['dominant_topic'] = np.argmax(all_topics_df.values, axis=1)
all_topics_df

Great! Now that we know the dominant topic in each abstract, we can try to visualise the most frequent words (i.e., **keywords**) across all abstracts with a specific predominant topic using a Wordcloud. To achieve this, we first need to identify which abstracts have the topic of interest as predominant topic.

In [ ]:
num_topic = 5
indices_abstracts_topic = list(all_topics_df[all_topics_df.dominant_topic == num_topic].index)
#clean_data_topic = np.array(clean_data)[indices_abstracts_topic] # this works in older versions of numpy
clean_data_topic = [clean_data[i] for i in indices_abstracts_topic]

In [ ]:
len(clean_data_topic), len(indices_abstracts_topic)

Great! Now we can extract the frequency of occurrence of each word in the abstracts of interest. To do this, we can generate again the `id2word` dictionary (word_id, word), derive the corpus and then calculate the overall count for each word across all the abstracts of interest.

In [ ]:
id2word = gensim.corpora.Dictionary(clean_data_topic)

corpus_topic = []
for abstract in clean_data_topic:
    corpus_topic.append(id2word.doc2bow(abstract))

In [ ]:
worddict = {}
for key, value in tqdm(id2word.items()):
    count = 0
    for abstract in corpus_topic:
        for (word_id, word_count) in abstract:
            if word_id == key:
                count = count + word_count
    worddict[value] =  count

The `worddict`is now a dictionary where each key is a word and the value is the number of times the word appears in the corpus. Now that we have this dictionary, we can use it directly to generate our wordcloud.

In [ ]:
wc = WordCloud(background_color="white", max_words=50)
wc.generate_from_frequencies(worddict)

_ = plt.figure(figsize = (10, 10))
plt.imshow(wc)
plt.axis("off")

Are the wordclouds genereated from the LDA words probabilities and word frequency very different? Can you use them to come up with the best description for each topic?

## Wordclouds of keywords

Finally, we can create a wordcloud using the frequency of occurrence of keywords in the abstracts with the predominat topic of interest. To achieve this, we can follow similar steps to the ones used to generate the wordcloud for the key words (sorry for the pun) from the abstracts. First, we need to convert the keywords column to list of words and extract only the abstracts of interest (e.g., abstracts with the topic of choice as predominant). Then, we can move forwards to count the frequency of occurrence of the words in a similar way to how we have done it before. 

In [ ]:
num_topic = 5
indices_abstracts_topic = list(all_topics_df[all_topics_df.dominant_topic == num_topic].index)
#clean_data_topic = np.array(clean_data)[indices_abstracts_topic] # this works in older versions of numpy
clean_data_topic = [clean_data[i] for i in indices_abstracts_topic]

data_df = pd.DataFrame.from_dict(data["documents"])
data_df.keywords = data_df.keywords.apply(lambda s: ast.literal_eval(s))
clean_data_topic_df = data_df.iloc[indices_abstracts_topic]
clean_data_topic_KW = clean_data_topic_df.keywords.tolist()

In [ ]:
id2word_KW = gensim.corpora.Dictionary(clean_data_topic_KW)

corpus_topic_KW = []
for abstract in clean_data_topic_KW:
    corpus_topic_KW.append(id2word_KW.doc2bow(abstract))

In [ ]:
worddict_KW = {}
for key, value in tqdm(id2word_KW.items()):
    count = 0
    for abstract in corpus_topic_KW:
        for (word_id, word_count) in abstract:
            if word_id == key:
                count = count + word_count
    worddict_KW[value] =  count

In [ ]:
wc = WordCloud(background_color="white", max_words=50)
wc.generate_from_frequencies(worddict_KW)

_ = plt.figure(figsize = (10, 10))
plt.imshow(wc)
plt.axis("off")

### Now extract a set of titles (from your authors/division/section/etc) and a list of keywords associated with a topic and use these as input to a conversational AI (ChatGPT, Gemini, Claude, etc): ask it create new titles in the style of the list of titles using the keywords provided (post your answers in the Teams chat - is it credible?)